# M3 実行報告・参照報告

最終更新: 2026-07-20 UTC

## 結論

| milestone | run ID | 判定 | checkpoint | certification |
|---|---|---|---:|---|
| M1 | `M1-20260719T235423Z-a7cacde2ead9` | 受理済み | 14 | `NOT_CERTIFIED` |
| M2 | `M2-20260720T005145Z-dd3e385d0a61` | 受理済み、M3へ進行可 | 14 | `NOT_CERTIFIED` |
| M3 | `M3-20260720T013551Z-ae995e91e861` | `CORE_REPRODUCED`、独立レビュー待ち | 14 | `NOT_CERTIFIED` |

M3 の GPU matrix-free Triad-ATRG pilot は実行完了し、全18 acceptance gate が PASS しました。固定 seed と checkpoint から core result を再現できる状態です。ただし M3 は探索 milestone であり、厳密な誤差 enclosure はありません。この notebook の参照確認と人間の独立レビューを終えるまで M4 へは進みません。


## M3 実測結果

- backend: `torch_cuda_opt_einsum`、NVIDIA RTX A4000、CUDA FP64、TF32 disabled
- operator: dimension 729、M2 の64 sector shard、production path は matrix-free
- matrix-free/explicit 最大絶対誤差: `1.5612511283791264e-17`
- adjoint 相対誤差: `3.552705686477058e-16`
- path cache: 7 entries、reuse PASS
- RSVD: rank 16、oversampling 16、power iteration 2、seed `20260720`
- RSVD Frobenius residual: `0.3441048053215534`
- RSVD relative residual: `0.23998764154389376`
- explicit optimal residual との比: `0.9999996365132263`
- Triad shape: left `729x16`、core `16x16`、right `16x729`
- influence proxy: `0.9999999999994724`、screening `INVESTIGATE_CUTOFF_AND_RANK`
- OOM retry: 0（OOM recovery 自体は test で PASS）
- CPU tests: 57 passed、5 deselected
- required GPU tests: 2 passed、9 deselected
- fresh-process resume / checkpoint basis restore / corrupt-newest fallback: PASS
- peak CPU memory: 約575.6 MiB
- peak GPU allocated memory: 37,013,504 bytes（約35.3 MiB）
- checkpoint: 398,702 bytes、save 0.092 s、verify 0.0024 s


## 人間が判断するときの要点

### 再現できたこと

- sector-shard matrix-free matvec と low-cutoff 明示行列の一致
- adjoint consistency
- 固定 seed RSVD basis の checkpoint 再現
- RSVD からの三因子 Triad 再構成
- contraction path cache と shard ordering の再利用・固定

### 証明していないこと

- RSVD residual の deterministic enclosure
- GPU 丸め誤差・backward error の bound
- influence proxy の rigorous spectral-radius bound
- cutoff `j2_max=1` を越える4D RG または mass gap

したがって適切な判定は「M3 core は再現できたが、M4開始前に独立レビューが必要」です。`CERTIFIED` と判定してはいけません。screening が `INVESTIGATE_CUTOFF_AND_RANK` なので、後続設計では cutoff/rank 感度を明示的に調べます。


## 参照先

- M3 report: `/storage/validated_4d_su2_rg/runs/M3-20260720T013551Z-ae995e91e861/reports/M3_report.json`
- M3 acceptance: `/storage/validated_4d_su2_rg/runs/M3-20260720T013551Z-ae995e91e861/reports/M3_acceptance.json`
- M3 manifest: `/storage/validated_4d_su2_rg/runs/M3-20260720T013551Z-ae995e91e861/run_manifest.json`
- M3 checkpoint: `/storage/validated_4d_su2_rg/runs/M3-20260720T013551Z-ae995e91e861/checkpoints/ckpt_000014`
- M3 実行 notebook: `notebooks/40_m3_gpu_triad_atrg.ipynb`
- M2 判定 notebook: `notebooks/35_m2_execution_reference_report.ipynb`
- M2→M3 受理記録: `audit/m2_accepted_parent.json`
- 実行ガイド: `README.md`


In [ ]:
from pathlib import Path
import hashlib
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_ROOT = Path('/storage/validated_4d_su2_rg/runs/M3-20260720T013551Z-ae995e91e861')
paths = {
    'M3 report': RUN_ROOT / 'reports/M3_report.json',
    'M3 acceptance': RUN_ROOT / 'reports/M3_acceptance.json',
    'M3 manifest': RUN_ROOT / 'run_manifest.json',
    'M3 checkpoint hashes': RUN_ROOT / 'checkpoints/ckpt_000014/hashes.json',
}
expected = {
    'M3 report': 'c3ffc856fa7d930ecc00bac96f1f69554950d33413b826fc92c8636540da9aa9',
    'M3 acceptance': 'e20104ef02ac90c6844a0a71ab993f8a6008872a19c6d47bfbe3a4d0a95f6179',
    'M3 manifest': '1f3aa0b27ced8930e897192b5ed7b41328498cb3555c220d25eae845edf0f26f',
    'M3 checkpoint hashes': 'df5a2eaf6fa63d3fd4c3ea37d9ba0da95a26c51b00c5bb5e8aabd43c3b6c75e3',
}
checks = {}
for label, path in paths.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None
    checks[label] = {'path': str(path), 'sha256': digest, 'PASS': digest == expected[label]}
if not all(item['PASS'] for item in checks.values()):
    raise RuntimeError('M3参照artifactが欠落または変更されています。M4を開始できません。')

report = json.loads(paths['M3 report'].read_text(encoding='utf-8'))
acceptance = json.loads(paths['M3 acceptance'].read_text(encoding='utf-8'))
required = {
    'report phase': report.get('phase') == 'M3_COMPLETE',
    'core reproduced': report.get('milestone_status') == 'CORE_REPRODUCED',
    'report not certified': report.get('certification_status') == 'NOT_CERTIFIED',
    'all report gates': all(report.get('acceptance_gates', {}).values()),
    'acceptance pass': acceptance.get('status') == 'PASS',
    'acceptance not certified': acceptance.get('certification_status') == 'NOT_CERTIFIED',
    'no rigorous bounds': report.get('rigorous_bounds') == [],
}
if not all(required.values()):
    raise RuntimeError(f'M3 fail-closed check failed: {required}')

matrix_free = report['results']['M3_MATRIX_FREE_VALIDATE']['result']
rsvd = report['results']['M3_RSVD']['result']
print(json.dumps({
    '判定': 'M3 CORE_REPRODUCED。NOT_CERTIFIED。M4開始には独立レビューが必要。',
    'artifact_checks': checks,
    'required_checks': required,
    'acceptance_gate_count': len(report['acceptance_gates']),
    'matrix_free_max_abs_error': matrix_free['matvec_max_abs_error'],
    'adjoint_relative_error': matrix_free['adjoint_relative_error'],
    'rsvd_relative_residual': rsvd['relative_residual_frobenius'],
    'influence_proxy': rsvd['influence_proxy'],
    'unresolved_issues': report['unresolved_issues'],
}, ensure_ascii=False, indent=2))


## 6時間停止後・別セッションからの再開

同じ M3 run を人間が再確認または再開する場合、fresh kernel で次を設定し、`notebooks/40_m3_gpu_triad_atrg.ipynb` を上から実行します。

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
export VALIDATED_RG_M2_RUN_ID=M2-20260720T005145Z-dd3e385d0a61
export VALIDATED_RG_M3_RUN_ID=M3-20260720T013551Z-ae995e91e861
```

orchestrator は最新の valid checkpoint を SHA-256 検証してロードします。最新 checkpoint が破損していれば一つ前の valid checkpoint へ戻り、`running` item は `pending` に戻します。15分周期と phase 境界で保存し、5時間20分までに final checkpoint を開始、5時間30分までに notebook へ制御を返します。すでに `M3_COMPLETE` の場合は再計算せず、保存済み結果を検証して表示します。
